In [2]:
"""
Fetch Solana yield pools from DefiLlama and save to CSV.

Run manually:
    python solana_pools.py

Or schedule it to refresh automatically (see notes at the bottom of this file).
"""

import csv
import requests
from datetime import datetime, timezone

URL = "https://yields.llama.fi/pools"
OUTPUT_FILE = "solana_pools.csv"

COLUMNS = [
    "Project",
    "Symbol",
    "APY",
    "APY Base",
    "APY Reward",
    "Reward Tokens",
    "Pool ID",
    "APY % 1D",
    "APY % 7D",
    "APY % 30D",
    "Stablecoin",
    "APY Base 7D",
    "TVL USD",
    "Underlying Tokens",
]


def get_solana_pools():
    response = requests.get(URL, timeout=30)
    response.raise_for_status()
    pools = response.json().get("data", [])

    rows = []
    for pool in pools:
        if pool.get("chain") == "Solana":
            rows.append({
                "Project": pool.get("project", ""),
                "Symbol": pool.get("symbol", ""),
                "APY": pool.get("apy", 0) or 0,
                "APY Base": pool.get("apyBase", 0) or 0,
                "APY Reward": pool.get("apyReward", 0) or 0,
                "Reward Tokens": ", ".join(pool.get("rewardTokens") or []),
                "Pool ID": pool.get("pool", ""),
                "APY % 1D": pool.get("apyPct1D", 0) or 0,
                "APY % 7D": pool.get("apyPct7D", 0) or 0,
                "APY % 30D": pool.get("apyPct30D", 0) or 0,
                "Stablecoin": pool.get("stablecoin", False),
                "APY Base 7D": pool.get("apyBase7d", 0) or 0,
                "TVL USD": pool.get("tvlUsd", 0) or 0,
                "Underlying Tokens": ", ".join(pool.get("underlyingTokens") or []),
            })

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(rows)

    print(f"[{datetime.now(timezone.utc).isoformat()}] "
          f"Wrote {len(rows)} Solana pools to {OUTPUT_FILE}")


if __name__ == "__main__":
    get_solana_pools()

# -----------------------------------------------------------------------
# Automating the refresh (pick one):
#
# 1) Windows Task Scheduler (simplest, no cloud needed):
#    - Create a Basic Task -> Trigger: Daily / hourly -> Action: Start a program
#    - Program: path to python.exe, Arguments: full path to this script
#
# 2) GitHub Actions (if you want the CSV hosted/versioned in a repo,
#    matches the automation pattern you already use for other pipelines):
#    - Put this script in a repo, add a workflow with a `schedule: cron`
#      trigger, run the script, then commit the updated CSV back
#      (or upload it as an artifact) each run.
# -----------------------------------------------------------------------

[2026-07-25T16:24:16.629394+00:00] Wrote 2901 Solana pools to solana_pools.csv


In [1]:
"""
Load solana_pools.csv into SQLite and let you query it with SQL,
with a separate "metadata" table for manually-added Image / App Link
columns (same idea as the hardcoded pool_address -> image/app-link
mapping in your DuneSQL query).

Flow:
  1. solana_pools.py refreshes solana_pools.csv (raw DefiLlama data).
  2. This script loads that CSV into the `pools` table every run
     (safe to overwrite - it's just refreshed data).
  3. Your manual Image / App Link additions live in `pool_metadata`,
     seeded from pool_metadata.csv the FIRST time only, then left
     alone (edits you make in the DB, or in that CSV + a re-seed,
     are never clobbered by a pools refresh).
  4. `pools_enriched` is a view that LEFT JOINs the two, same shape
     as your `defi_combined` query.

Usage:
    python solana_pools_db.py                  # load/refresh + rebuild view
    sqlite3 solana_pools.db                     # open a shell
    sqlite> SELECT * FROM pools_enriched WHERE "TVL USD" > 0 ORDER BY Project, "TVL USD" DESC;
"""

import csv
import os
import sqlite3

DB_FILE = "solana_pools.db"
POOLS_CSV = "solana_pools.csv"
METADATA_CSV = "pool_metadata.csv"  # you maintain this by hand


def load_pools(conn):
    """(Re)load raw pool data. Safe to overwrite - it's just a refresh."""
    with open(POOLS_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        columns = reader.fieldnames

    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS pools")
    col_defs = ", ".join(f'"{c}" TEXT' for c in columns)
    cur.execute(f"CREATE TABLE pools ({col_defs})")

    placeholders = ", ".join("?" for _ in columns)
    cur.executemany(
        f"INSERT INTO pools VALUES ({placeholders})",
        [[row[c] for c in columns] for row in rows],
    )
    conn.commit()
    print(f"Loaded {len(rows)} rows into 'pools'.")


def ensure_metadata_table(conn):
    """Create pool_metadata once. Seed from CSV only if the table is new."""
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS pool_metadata (
            "Pool ID" TEXT PRIMARY KEY,
            "Image" TEXT,
            "App Link" TEXT
        )
        """
    )
    conn.commit()

    cur.execute("SELECT COUNT(*) FROM pool_metadata")
    is_empty = cur.fetchone()[0] == 0

    if is_empty and os.path.exists(METADATA_CSV):
        with open(METADATA_CSV, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = [(r["Pool ID"], r.get("Image", ""), r.get("App Link", "")) for r in reader]
        cur.executemany(
            'INSERT OR IGNORE INTO pool_metadata VALUES (?, ?, ?)', rows
        )
        conn.commit()
        print(f"Seeded pool_metadata with {len(rows)} rows from {METADATA_CSV}.")
    elif is_empty:
        # Create an empty template CSV so you know the expected columns.
        with open(METADATA_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["Pool ID", "Image", "App Link"])
        print(f"No metadata yet - created a blank template at {METADATA_CSV}.")


def create_enriched_view(conn):
    cur = conn.cursor()
    cur.execute("DROP VIEW IF EXISTS pools_enriched")
    cur.execute(
        """
        CREATE VIEW pools_enriched AS
        SELECT p.*, m."Image", m."App Link"
        FROM pools p
        LEFT JOIN pool_metadata m ON p."Pool ID" = m."Pool ID"
        """
    )
    conn.commit()


def main():
    conn = sqlite3.connect(DB_FILE)
    load_pools(conn)
    ensure_metadata_table(conn)
    create_enriched_view(conn)
    conn.close()
    print(f"Ready. Query {DB_FILE} -> table 'pools_enriched'.")


if __name__ == "__main__":
    main()

# -----------------------------------------------------------------------
# To add Image / App Link for specific pools, either:
#
#   A) Edit rows directly:
#      sqlite3 solana_pools.db
#      sqlite> INSERT OR REPLACE INTO pool_metadata VALUES
#          ('d8733ab8-a147-4e31-a668-2c9dff24ea56',
#           'https://pbs.twimg.com/.../image.jpg',
#           'https://someapp.xyz/pool/...');
#
#   B) Edit pool_metadata.csv (Pool ID, Image, App Link columns) and
#      re-seed by deleting pool_metadata.db's rows, or just:
#      sqlite> DELETE FROM pool_metadata;   -- then re-run this script
#
# Either way, `pools_enriched` always reflects the latest join of
# fresh pool data + your manual metadata.
# -----------------------------------------------------------------------

Loaded 2901 rows into 'pools'.
No metadata yet - created a blank template at pool_metadata.csv.
Ready. Query solana_pools.db -> table 'pools_enriched'.


In [8]:
import sqlite3

conn = sqlite3.connect("solana_pools.db")
conn.row_factory = sqlite3.Row  # <-- this line does it
cur = conn.cursor()

cur.execute("SELECT * FROM pools_enriched LIMIT 5")
for row in cur.fetchall():
    print(dict(row))

{'Project': 'binance-staked-sol', 'Symbol': 'BNSOL', 'APY': '4.7782', 'APY Base': '4.7782', 'APY Reward': '0', 'Reward Tokens': '', 'Pool ID': '9e709e57-84eb-496b-82ce-2e8f6a17db1b', 'APY % 1D': '-0.06135', 'APY % 7D': '-0.1271', 'APY % 30D': '-0.31119', 'Stablecoin': 'False', 'APY Base 7D': '0', 'TVL USD': '756970273', 'Underlying Tokens': 'So11111111111111111111111111111111111111112', 'Image': None, 'App Link': None}
{'Project': 'jito-liquid-staking', 'Symbol': 'JITOSOL', 'APY': '5.17', 'APY Base': '5.17', 'APY Reward': '0', 'Reward Tokens': '', 'Pool ID': '0e7d0722-9054-4907-8593-567b353c0900', 'APY % 1D': '-0.01', 'APY % 7D': '-0.1', 'APY % 30D': '0.06', 'Stablecoin': 'False', 'APY Base 7D': '0', 'TVL USD': '744980856', 'Underlying Tokens': 'So11111111111111111111111111111111111111112', 'Image': None, 'App Link': None}
{'Project': 'blackrock-buidl', 'Symbol': 'BUIDL', 'APY': '3.5379', 'APY Base': '3.5379', 'APY Reward': '0', 'Reward Tokens': '', 'Pool ID': '590d770e-ed5d-4c8d-ad96-

In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("solana_pools.db")

df = pd.read_sql_query("SELECT * FROM pools_enriched LIMIT 5", conn)
print(df.to_string(index=False))

conn.close()

            Project  Symbol     APY APY Base APY Reward                                Reward Tokens                              Pool ID APY % 1D APY % 7D APY % 30D Stablecoin APY Base 7D   TVL USD                            Underlying Tokens Image App Link
 binance-staked-sol   BNSOL  4.7782   4.7782          0                                              9e709e57-84eb-496b-82ce-2e8f6a17db1b -0.06135  -0.1271  -0.31119      False           0 756970273  So11111111111111111111111111111111111111112  None     None
jito-liquid-staking JITOSOL    5.17     5.17          0                                              0e7d0722-9054-4907-8593-567b353c0900    -0.01     -0.1      0.06      False           0 744980856  So11111111111111111111111111111111111111112  None     None
    blackrock-buidl   BUIDL  3.5379   3.5379          0                                              590d770e-ed5d-4c8d-ad96-5178c2072295   -3e-05  0.05572   0.02415       True           0 653839521 GyWgeqpy5GueU2YbkE8xqUeV

In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("solana_pools.db")

df = pd.read_sql_query("SELECT COUNT(DISTINCT Project) FROM pools_enriched", conn)
print(df.to_string(index=False))

conn.close()

 COUNT(DISTINCT Project)
                      52
